# 01 - Exploration du Dataset (Brain Tumor Detection)

Ce notebook a pour objectif d'explorer la structure du dataset d'images IRM de tumeurs cérébrales, de vérifier l'équilibre des classes (`yes` / `no`), d'inspecter les dimensions et formats des images, et de visualiser quelques échantillons.

In [3]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Configuration de l'affichage
%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Configuration des chemins et comptage des images
> **⚠️ IMPORTANT :** Modifiez la variable `DATA_DIR` ci-dessous si le dossier de votre dataset se trouve à un autre emplacement.

In [ ]:
# Variable configurable pour le chemin du dataset
DATA_DIR = "brain_tumor_dataset"  # À adapter selon votre arborescence (ex: "../data/raw/brain_tumor_dataset")

yes_path = os.path.join(DATA_DIR, "yes")
no_path = os.path.join(DATA_DIR, "no")

# Vérification de l'existence des dossiers
if not os.path.exists(yes_path) or not os.path.exists(no_path):
    print(f"⚠️ Attention : Le dossier '{DATA_DIR}' ou ses sous-dossiers 'yes'/'no' sont introuvables. Veuillez vérifier le chemin.")
else:
    yes_files = os.listdir(yes_path)
    no_files = os.listdir(no_path)
    
    print(f"Nombre d'images avec tumeur (yes) : {len(yes_files)}")
    print(f"Nombre d'images sans tumeur (no) : {len(no_files)}")
    print(f"Total des images dans le dataset : {len(yes_files) + len(no_files)}")

## 2. Distribution des classes (Équilibre du dataset)
Visualisons la répartition entre les images positives (tumeur détectée) et négatives.

In [ ]:
if 'yes_files' in locals() and 'no_files' in locals():
    df_dist = pd.DataFrame({
        'Classe': ['Tumeur (Yes)', 'Sain (No)'],
        'Nombre': [len(yes_files), len(no_files)]
    })

    plt.figure(figsize=(6, 4))
    sns.barplot(x='Classe', y='Nombre', data=df_dist, palette='viridis')
    plt.title("Distribution des classes du Dataset")
    plt.ylabel("Nombre d'images")
    plt.show()

## 3. Visualisation d'exemples d'images
Affichage aléatoire de quelques échantillons de chaque classe pour inspecter visuellement les IRM.

In [ ]:
import random

def plot_sample_images(folder_path, label, n=4):
    if not os.path.exists(folder_path):
        return
    files = os.listdir(folder_path)
    sample_files = random.sample(files, min(n, len(files)))
    
    plt.figure(figsize=(12, 3))
    for i, file in enumerate(sample_files):
        img_path = os.path.join(folder_path, file)
        img = Image.open(img_path)
        
        plt.subplot(1, n, i + 1)
        plt.imshow(img, cmap='gray' if img.mode == 'L' else None)
        plt.title(f"{label} - {file}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()

if 'yes_path' in locals() and os.path.exists(yes_path):
    print("Échantillons de la classe 'yes' (Tumeur) :")
    plot_sample_images(yes_path, "Yes", n=4)

if 'no_path' in locals() and os.path.exists(no_path):
    print("Échantillons de la classe 'no' (Sain) :")
    plot_sample_images(no_path, "No", n=4)

## 4. Analyse des dimensions et des formats d'images
Vérifions si les images ont toutes la même taille (résolution) et quels sont leurs modes de couleur (niveaux de gris vs RGB).

In [ ]:
def analyze_image_properties(folder_path):
    if not os.path.exists(folder_path):
        return [], [], []
    widths, heights, modes = [], [], []
    for file in os.listdir(folder_path):
        img_path = os.path.join(folder_path, file)
        try:
            with Image.open(img_path) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)
                modes.append(img.mode)
        except Exception as e:
            print(f"Erreur avec le fichier {file}: {e}")
    return widths, heights, modes

if os.path.exists(yes_path) and os.path.exists(no_path):
    w_yes, h_yes, m_yes = analyze_image_properties(yes_path)
    w_no, h_no, m_no = analyze_image_properties(no_path)

    print(f"Classe YES - Largeurs min/max : {min(w_yes)} / {max(w_yes)} | Hauteurs min/max : {min(h_yes)} / {max(h_yes)}")
    print(f"Classe YES - Modes couleurs : {set(m_yes)}")
    print(f"Classe NO  - Largeurs min/max : {min(w_no)} / {max(w_no)} | Hauteurs min/max : {min(h_no)} / {max(h_no)}")
    print(f"Classe NO  - Modes couleurs : {set(m_no)}")
    print("\nConclusion : Les images possèdent des tailles variées. Une étape de redimensionnement sera requise dans le preprocessing (Notebook 02).")